# Part 4 — Gemini, structured follow-up notes and recommendations
Requires a free Gemini API key entered at runtime. Never paste the key into a code cell or upload it.

In [ ]:
# In Colab, run this cell once:
%pip install -q google-genai pydantic
import json, time, getpass
from pathlib import Path
import pandas as pd
from pydantic import BaseModel, Field, ValidationError, field_validator
from google import genai
from google.genai import types
ROOT=Path.cwd()
notes=pd.read_csv(ROOT/'data'/'followup_notes.csv').head(10)
API_KEY=getpass.getpass('Gemini API key (hidden): ')
client=genai.Client(api_key=API_KEY)
del API_KEY
MODEL_NAME='gemini-2.5-flash' # Change to the current Flash model listed in your AI Studio account if needed.
api_calls=0
ALLOWED=['forgot_or_no_reminder','distance_or_transport','cost_or_fee',
         'illness_or_emergency','work_or_schedule','other']
SCHEMA={'type':'object','properties':{'root_cause':{'type':'string'},
         'confidence':{'type':'number'},'summary':{'type':'string'},
         'recommended_action':{'type':'string'}},
        'required':['root_cause','confidence','summary','recommended_action']}
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0,le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)
    @field_validator('root_cause')
    @classmethod
    def allowed_root_cause(cls,v):
        if v not in ALLOWED: raise ValueError(f'root_cause must be one of {ALLOWED}')
        return v


## 4.1 One validated call using CRAFT

In [ ]:
SYSTEM_TEXT=('You are a careful patient follow-up analyst. Return only valid JSON with the four '
             'required fields. Never infer unsupported facts. Summaries must be at most 25 words.')
def craft_prompt(note_text):
    return (f'Context: A patient missed an outpatient appointment.\n'
            f'Role: Classify the call-centre note for clinic operations.\n'
            f'Action: Identify the single best supported root cause, confidence 0 to 1, '
            f'a summary of at most 25 words, and a practical recommended action.\n'
            f'Tone: Neutral, factual, concise.\n'
            f'Format: JSON with root_cause, confidence, summary, recommended_action.\n'
            f'Allowed root_cause values: {", ".join(ALLOWED)}. '
            f'If unclear, choose other rather than guess.\nNote: {note_text}')
def analyse_note(note_text, crafted=True):
    global api_calls
    prompt=craft_prompt(note_text) if crafted else f'Classify this note: {note_text}'
    config=types.GenerateContentConfig(system_instruction=SYSTEM_TEXT,temperature=0,
        max_output_tokens=1200,response_mime_type='application/json',response_schema=SCHEMA,
        thinking_config=types.ThinkingConfig(thinking_budget=0))
    # Some current Flash models do not support this thinking configuration.
    # Only remove thinking_config if the API explicitly rejects it.
    api_calls+=1
    response=client.models.generate_content(model=MODEL_NAME,contents=prompt,config=config)
    return json.loads(response.text),response.text
first,raw=analyse_note(str(notes.iloc[0]['note_text']))
print('N001:',first)
print('Pydantic validated:',FollowupInsight(**first).model_dump())


## 4.2 All first ten notes, including blank note N004; preserve invalid raw responses

In [ ]:
records=[]
for i,row in enumerate(notes.itertuples(index=False)):
    entry={'note_id':row.note_id,'appointment_id':row.appointment_id,
           'root_cause':None,'confidence':None,'summary':None,
           'recommended_action':None,'validation_status':None,'raw_response':''}
    if pd.isna(row.note_text) or not str(row.note_text).strip():
        entry.update(root_cause='other',confidence=0.0,summary='No note recorded',
                     recommended_action='Re-call patient',validation_status='skipped_blank')
    else:
        try:
            data,raw=analyse_note(str(row.note_text))
            entry['raw_response']=raw
            for field in ('root_cause','confidence','summary','recommended_action'):
                entry[field]=data.get(field)
            FollowupInsight(**data)
            if len(str(data['summary']).split())>25: raise ValueError('Summary exceeds 25 words')
            entry['validation_status']='valid'
        except (ValidationError,ValueError,KeyError,TypeError,Exception) as exc:
            # Never repair invalid model content; preserve raw text if returned.
            entry['validation_status']='invalid'
            print('Invalid response / API error for',row.note_id,':',repr(exc))
    records.append(entry)
    if i < len(notes)-1 and entry['validation_status']!='skipped_blank': time.sleep(13)
columns=['note_id','appointment_id','root_cause','confidence','summary',
         'recommended_action','validation_status','raw_response']
analysed=pd.DataFrame(records,columns=columns)
analysed.to_csv(ROOT/'notes_analysed.csv',index=False)
print(analysed[['note_id','root_cause','validation_status']].to_string(index=False))
print('API calls so far:',api_calls)
assert analysed.shape==(10,8)


## 4.3 Deliberately invalid Pydantic examples and two prompt comparisons

In [ ]:
good={'root_cause':'other','confidence':0.5,'summary':'The reason is unclear.',
      'recommended_action':'Call patient for clarification'}
for bad in [dict(good,confidence=1.7),{k:v for k,v in good.items() if k!='recommended_action'}]:
    try: FollowupInsight(**bad)
    except ValidationError as exc: print('Expected validation error:',exc)
comparison=[]
for note_id in ['N007','N009']:
    text=notes.loc[notes['note_id']==note_id,'note_text'].iloc[0]
    for variant in ['V1','V2']:
        try:
            data,raw=analyse_note(str(text),crafted=(variant=='V2'))
            try:
                FollowupInsight(**data)
                valid=True
            except ValidationError: valid=False
            comparison.append({'note_id':note_id,'prompt':variant,
                               'root_cause':data.get('root_cause'),'allowed':data.get('root_cause') in ALLOWED,
                               'valid':valid})
        except Exception as exc:
            comparison.append({'note_id':note_id,'prompt':variant,'root_cause':None,
                               'allowed':False,'valid':False})
            print('API error:',exc)
        time.sleep(13)
print(pd.DataFrame(comparison).to_string(index=False))
print('Total API calls across Tasks 4.1–4.3:',api_calls)
print('Prefer V2 if it returns allowed, evidence-based categories without guessing.')


## 4.4 Cross-tab and evidence for README

In [ ]:
appointments=pd.read_csv(ROOT/'appointments_clean.csv')
merged=analysed.merge(appointments,on='appointment_id',how='left',validate='one_to_one')
assert len(merged)==10
print(pd.crosstab(merged['root_cause'],merged['reminder_sent'],dropna=False))
print('Notes with no reminder:',int((merged['reminder_sent']=='No').sum()))
print('Valid classifications:',int((analysed['validation_status']=='valid').sum()))
print('Use actual notebook outputs to complete README; do not assume the LLM outputs.')
